In [1]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# 设置全局缓存路径（对所有Hugging Face组件生效）
os.environ["TRANSFORMERS_CACHE"] = "/home/chelly/disk1/huggingface_cache/"
os.makedirs(os.environ["TRANSFORMERS_CACHE"], exist_ok=True)

# os.environ["http_proxy"] = "http://10.147.20.23:7897"
# os.environ["https_proxy"] = "http://10.147.20.23:7897"
# os.environ["http_proxy"] = "http://192.168.1.8:7897"
# os.environ["https_proxy"] = "http://192.168.1.8:7897"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
import numpy as np
import torch
import cv2 # 导入OpenCV库
import re
from PIL import Image
from transformers.image_transforms import resize
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
import json

# --- 定义一个函数来读取视频帧，替代 read_video_pyav ---
def load_video(video_path, num_segments=8):
    """
    从视频文件或图片帧文件夹加载帧。
    处理灰度图和RGBA图，将其转换为3通道RGB格式。
    对于图片文件夹，优先使用 Pillow 读取图片以提高兼容性。

    Args:
        video_path (str): 视频文件的路径 (例如 .mp4) 或
                          包含按序编号的图片帧的目录 (例如 .png, .jpg)。
        num_segments (int): 目标要从视频/图片中均匀采样的帧数。
                            如果总帧数少于 num_segments，则获取所有可用帧。

    Returns:
        np.ndarray: 一个包含加载的视频帧的NumPy数组 (形状: N, 高度, 宽度, 3)，
                    其中 N 为 min(总帧数, num_segments)。
                    帧保证为3通道RGB格式。

    Raises:
        ValueError: 如果 video_path 无效、目录中未找到图片或未加载任何帧。
        IOError: 如果无法打开视频文件。
    """
    if not os.path.exists(video_path):
        raise ValueError(f"路径不存在: {video_path}")

    if num_segments <= 0:
        print("警告: num_segments 必须大于0。返回空数组。")
        return np.array([])

    frames_list = []
    total_frames = 0

    if os.path.isdir(video_path):
        # 情况1: video_path 是一个图片帧目录
        image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp') # 常见的图片扩展名
        all_image_files = []
        for f_name in os.listdir(video_path):
            if f_name.lower().endswith(image_extensions):
                all_image_files.append(os.path.join(video_path, f_name))

        if not all_image_files:
            raise ValueError(f"在目录中未找到任何图片文件: {video_path}。支持的格式: {', '.join(image_extensions)}")

        # 根据文件名中的数字部分对图片文件进行排序，确保帧的顺序
        def numerical_sort_key(value):
            # 提取字符串中的所有数字并转换为整数，用于排序
            parts = re.findall(r'(\d+)', value)
            return [int(part) for part in parts]

        all_image_files.sort(key=numerical_sort_key)
        total_frames = len(all_image_files)

        # 确定要采样的索引
        if total_frames > num_segments:
            # 使用 np.linspace 均匀采样 num_segments 帧
            selected_indices = np.linspace(0, total_frames - 1, num_segments).astype(int)
        else:
            # 如果总帧数少于 num_segments，则获取所有可用帧
            selected_indices = np.arange(0, total_frames).astype(int)

        # 读取选定的图片帧
        for idx in selected_indices:
            if idx >= len(all_image_files): # 确保索引在范围内
                continue

            img_path = all_image_files[idx]
            frame = None # 初始化frame

            try:
                # 使用 Pillow 读取图像
                with Image.open(img_path) as pil_img:
                    # 将图像转换为 RGB 模式，以确保一致性 (处理灰度、调色板等)
                    # 对于L（灰度）和P（调色板）模式，转换为RGB
                    if pil_img.mode != 'RGB' and pil_img.mode != 'RGBA':
                        pil_img = pil_img.convert('RGB') # 转换为3通道RGB
                    
                    frame = np.array(pil_img) # 转换为 NumPy 数组

                    # Pillow 读取的图像是 RGB 格式，而 OpenCV 处理的图像通常是 BGR
                    # 但我们最终想要的是 RGB 输出，所以如果 Pillow 已经是 RGB/RGBA，不需要额外转换
                    if frame.ndim == 2: # 如果 Pillow 最终返回了灰度图 (L模式转RGB后通常不会，但以防万一)
                        frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)
                    elif pil_img.mode == 'RGBA': # 如果是 RGBA，去除 alpha 通道，转换为 RGB
                        frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2RGB)
                    # 如果 pil_img.mode 是 'RGB'，此时 frame 已经是 (H,W,3) 且是 RGB 顺序，无需转换
            except Exception as e:
                print(f"警告: Pillow 无法读取图片文件: {img_path}. 错误: {e}. 跳过此帧。")
                continue # 跳过无法读取的帧

            if frame is not None:
                frames_list.append(frame)

    else:
        # 情况2: video_path 是一个视频文件 (继续使用 OpenCV)
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"无法打开视频文件: {video_path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            cap.release()
            raise ValueError(f"视频文件 {video_path} 不包含任何帧。")

        # 确定要采样的索引
        if total_frames > num_segments:
            selected_indices = np.linspace(0, total_frames - 1, num_segments).astype(int)
        else:
            selected_indices = np.arange(0, total_frames).astype(int)

        for target_idx in selected_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
            ret, frame = cap.read()

            if ret:
                # OpenCV 默认读取的是 BGR，转换为 RGB
                if frame.ndim == 2: # 灰度视频帧
                    frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)
                elif frame.shape[2] == 4: # RGBA 视频帧
                    frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2RGB)
                else: # 假设是3通道BGR
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames_list.append(frame)
            # else: 如果 ret 为 False，表示无法读取帧，通常是到达视频末尾或索引无效
            # 这里的 selected_indices 已经确保了索引的有效性，所以通常不会出现这种情况
            # 如果出现，说明视频可能损坏，但我们继续尝试读取其他帧。

        cap.release() # 释放视频捕获对象

    if not frames_list:
        raise ValueError(f"未能从 {video_path} 加载任何帧。请检查路径和文件内容。")

    return np.array(frames_list)

def pre_process_video(images: np.ndarray, size = (384, 384)):
    images = [
                resize(image=image, size=size)
                for image in images
            ]
    images = np.stack(images, axis=0)
    return images

def denormalize(tensor, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]):
    """
    Denormalizes an image tensor.

    This function reverses the normalization process, converting a tensor with a
    specific mean and standard deviation back to a tensor with values in the [0, 1] range.

    Args:
        tensor (torch.Tensor): The input tensor to denormalize. 
                               Assumes shape (C, H, W) or (B, C, H, W).
        mean (list or tuple): The mean used for normalization.
        std (list or tuple): The standard deviation used for normalization.

    Returns:
        torch.Tensor: The denormalized tensor with values clamped to [0, 1].
    """
    # Clone the tensor to avoid modifying the original in-place
    denorm_tensor = torch.squeeze(tensor.clone(), dim=0)

    # Ensure mean and std are tensors and on the same device and dtype as the input
    mean = torch.tensor(mean, device=denorm_tensor.device, dtype=denorm_tensor.dtype)
    std = torch.tensor(std, device=denorm_tensor.device, dtype=denorm_tensor.dtype)

    # Reshape mean and std to (C, 1, 1) so they can be broadcasted across the image dimensions
    mean = mean.view(-1, 1, 1)
    std = std.view(-1, 1, 1)
    
    # Handle both single image (C, H, W) and batch of images (B, C, H, W)
    if denorm_tensor.dim() == 4:
        # Unsqueeze mean and std to (1, C, 1, 1) for batch broadcasting
        mean = mean.unsqueeze(0)
        std = std.unsqueeze(0)

    # Perform the denormalization: (tensor * std) + mean
    denorm_tensor.mul_(std).add_(mean)

    # Clamp the values to be in the valid [0, 1] range
    # denorm_tensor = torch.clamp(denorm_tensor, 0, 1)

    return denorm_tensor

def tensor_to_pil(tensor, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]):
    """
    Converts a normalized tensor to a PIL Image for visualization.
    
    Args:
        tensor (torch.Tensor): The input tensor, expects shape (C, H, W).
                               If the tensor is on a GPU, it will be moved to the CPU.
        mean (list or tuple): The mean used for normalization.
        std (list or tuple): The standard deviation used for normalization.

    Returns:
        PIL.Image.Image: The resulting image.
    """
    np_image = tensor_to_numpy(tensor, mean=mean, std=std)
    
    pil_image = Image.fromarray(np_image)
    
    return pil_image

def tensor_to_numpy(tensor, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]):
    """
    Converts a normalized tensor to a PIL Image for visualization.
    
    Args:
        tensor (torch.Tensor): The input tensor, expects shape (C, H, W).
                               If the tensor is on a GPU, it will be moved to the CPU.
        mean (list or tuple): The mean used for normalization.
        std (list or tuple): The standard deviation used for normalization.

    Returns:
        np.array: The resulting numpy.array.
    """
    # 1. Denormalize the tensor
    # Ensure tensor is float32 for denormalization calculations
    denormalized_tensor = denormalize(tensor.cpu().to(torch.float32), mean, std)
    
    # 2. Convert to NumPy array and scale to [0, 255]
    # Permute dimensions from (C, H, W) to (H, W, C) for PIL
    np_image = denormalized_tensor.numpy()
    np_image = np.transpose(np_image, (1, 2, 0))
    np_image = (np_image * 255).astype(np.uint8)
    
    return np_image

def show_pil_images(pil_images: Image.Image):
    """
    Display a list of PIL images in two rows using matplotlib.

    Args:
        pil_images (list or tuple): List of PIL.Image.Image objects.
    """

    if not isinstance(pil_images, (list, tuple)):
        pil_images = [pil_images]

    num_images = len(pil_images)
    if num_images == 0:
        print("No images to display.")
        return

    # Determine number of rows and columns (2 rows)
    n_rows = 2
    n_cols = (num_images + 1) // 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten() if num_images > 1 else [axes]

    for idx, img in enumerate(pil_images):
        axes[idx].imshow(img)
        axes[idx].axis('off')
    # Hide any unused subplots
    for idx in range(num_images, n_rows * n_cols):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

def show_numpy_images(numpy_images: np.ndarray):
    """
    Display a list of numpy images in two rows using matplotlib.

    Args:
        numpy_images (np.ndarray): A numpy array of images.
    """

    if not isinstance(numpy_images, np.ndarray):
        raise ValueError("Input must be a numpy array.")

    # If input is (N, C, H, W), convert to (N, H, W, C)
    if numpy_images.ndim == 4 and numpy_images.shape[1] in [1, 3, 4]:
        # Assume (N, C, H, W)
        numpy_images = np.transpose(numpy_images, (0, 2, 3, 1))

    num_images = numpy_images.shape[0]
    if num_images == 0:
        print("No images to display.")
        return

    n_rows = 2
    n_cols = (num_images + 1) // 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten() if num_images > 1 else [axes]

    for idx in range(num_images):
        img = numpy_images[idx]
        # If image is float, clip and convert to uint8 for display
        if img.dtype != np.uint8:
            img = np.clip(img, 0, 1)
            img = (img * 255).astype(np.uint8)
        axes[idx].imshow(img)
        axes[idx].axis('off')
    # Hide any unused subplots
    for idx in range(num_images, n_rows * n_cols):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

def load_attack_video(path):
    attack_tensor = torch.load(path).squeeze(0)
    attack_numpy = np.stack([tensor_to_numpy(tensor) for tensor in attack_tensor], axis=0)
    return attack_numpy

def get_question(question_path, video_name):
    # Load question
    question = defaultdict(list)
    question_path = Path(question_path)
    if question_path.exists():
        with open(question_path) as f:
            meta_question = json.load(f)
        key_name = 'video_name'
        feat_name = [k for k in meta_question[0].keys() if k != key_name]
        for item in meta_question:
            question[item[key_name]].append({f: item[f] for f in feat_name})  # video_name: [{question1, question_id1}, ...]
    return question[video_name][0]["question"]


/opt/conda/envs/vlma3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/vlma3/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
import torch.nn.functional as F
from typing import Dict, List, Tuple, Optional

class VideoGradCAM:
    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        self.hooks = []
        self.gradients = {}
        self.activations = {}

    def register_hooks(self):
        """Register hooks to capture gradients and activations."""
        def get_activation(name):
            def hook(module, input, output):
                if hasattr(output, 'last_hidden_state'):
                    self.activations[name] = output.last_hidden_state.detach().cpu()
                elif hasattr(output, 'hidden_states') and output.hidden_states is not None:
                    self.activations[name] = output.hidden_states[-1].detach().cpu()
                elif isinstance(output, torch.Tensor):
                    self.activations[name] = output.detach().cpu()
                else:
                    try:
                        self.activations[name] = output.detach().cpu()
                    except:
                        print(f"Warning: Could not extract activation for {name}, output type: {type(output)}")
            return hook

        def get_gradient(name):
            def hook(module, grad_input, grad_output):
                if grad_output[0] is not None:
                    self.gradients[name] = grad_output[0].detach().cpu()
            return hook

        # Register only the hooks needed for GradCAM
        vision_hook = self.model.vision_tower.register_forward_hook(
            get_activation('vision_features'))
        vision_grad_hook = self.model.vision_tower.register_backward_hook(
            get_gradient('vision_features'))

        projector_hook = self.model.multi_modal_projector.register_forward_hook(
            get_activation('projected_features'))
        projector_grad_hook = self.model.multi_modal_projector.register_backward_hook(
            get_gradient('projected_features'))

        self.hooks.extend([vision_hook, vision_grad_hook, projector_hook, projector_grad_hook])

    def remove_hooks(self):
        """Remove all hooks and clear stored gradients/activations."""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
        self.gradients = {}
        self.activations = {}

    def compute_video_gradcam(
        self,
        video: torch.Tensor,
        question: str,
        target_strategy: str = "first_generated",
        target_answer: Optional[str] = None,
        normalize_method: str = "per_frame"
    ) -> Tuple[torch.Tensor, str]:
        """
        GradCAM calculation for video QA models.

        Args:
            target_strategy: 
                - "first_generated": Only the first generated token's gradient.
                - "last_generated": Full answer, gradient of last token.
                - "answer_average": Full answer, average gradient over all tokens.
        """
        self.register_hooks()
        try:
            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": question},
                        {"type": "video"},
                    ],
                },
            ]
            prompt = self.processor.apply_chat_template(conversation, add_generation_prompt=True)

            # Ensure video is torch tensor and correct shape
            if isinstance(video, np.ndarray):
                video = torch.from_numpy(video).float()
            if video.shape[-1] == 3:
                video = video.permute(0, 3, 1, 2)

            # Prepare input on CPU first, then move only needed tensors to GPU
            # --- FIX: input_ids must be Long, not float16 ---
            # inputs_base = self.processor(text=prompt, videos=[video], padding=True, return_tensors="pt")
            # if "pixel_values_videos" in inputs_base:
            #     inputs_base["pixel_values_videos"] = inputs_base["pixel_values_videos"].to(self.model.device, self.model.dtype)
            inputs_base = self.processor(text=prompt, videos=[video], padding=True, return_tensors="pt").to(
                self.model.device, self.model.dtype)

            print(f"Base input sequence length: {inputs_base['input_ids'].shape[1]}")

            # No mixed precision: all operations in default precision
            if target_strategy == "first_generated":
                return self._compute_first_token_gradcam(inputs_base, video, normalize_method)
            elif target_strategy == "last_generated":
                return self._compute_last_token_gradcam(inputs_base, video, normalize_method)
            elif target_strategy == "answer_average":
                return self._compute_average_token_gradcam(inputs_base, video, normalize_method)
            else:
                raise ValueError(f"Unknown target_strategy: {target_strategy}")

        except Exception as e:
            print(f"Error in compute_video_gradcam: {e}")
            import traceback
            traceback.print_exc()
            num_frames = video.shape[0] if len(video.shape) >= 4 else 1
            height, width = video.shape[-2:] if len(video.shape) >= 3 else (384, 384)
            gradcam_maps = torch.zeros(num_frames, height, width)
            return gradcam_maps, f"Error: {str(e)}"
        finally:
            self.remove_hooks()
            torch.cuda.empty_cache()

    def _compute_first_token_gradcam(self, inputs_base, video, normalize_method):
        """Compute GradCAM for the first generated token with minimal memory."""
        print("Computing GradCAM for first generated token...")
        with torch.no_grad():
            outputs_gen = self.model.generate(
                **inputs_base, 
                max_new_tokens=1,  # 只生成1个token
                do_sample=False,   # 使用贪心搜索，确保与推理一致
                pad_token_id=self.processor.tokenizer.eos_token_id
            )
        
        # 获取生成的token
        input_length = inputs_base['input_ids'].shape[1]
        if outputs_gen.shape[1] > input_length:
            generated_token_id = outputs_gen[0, input_length].item()
            generated_text = self.processor.tokenizer.decode([generated_token_id], num_logits_to_keep=True)
        else:
            print("Warning: No token generated, using fallback")
            generated_token_id = self.processor.tokenizer.encode("The")[0]
            generated_text = "The"
        print(f"First generated token: {generated_text} (ID: {generated_token_id})")
        # Free memory from generation
        del outputs_gen
        torch.cuda.empty_cache()
        return self._compute_gradcam_for_position(
            inputs_base, 0, generated_token_id, normalize_method, generated_text, 1
        )

    def _compute_last_token_gradcam(self, inputs_base, video, normalize_method):
        """Compute GradCAM for the last generated token with minimal memory."""
        print("Computing GradCAM for last generated token...")
        with torch.no_grad():
            outputs_gen = self.model.generate(
                **inputs_base,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=self.processor.tokenizer.eos_token_id
            )
        input_length = inputs_base['input_ids'].shape[1]
        if outputs_gen.shape[1] > input_length:
            generated_tokens = outputs_gen[0, input_length:]
            generated_text = self.processor.tokenizer.decode(generated_tokens, num_logits_to_keep=True)
            
            # 使用最后一个生成的token
            last_token_id = generated_tokens[-1]
            print(f"Generated text: {generated_text}")
            print(f"Last token: {self.processor.tokenizer.decode(last_token_id)} (ID: {last_token_id})")
            
            # 构建到最后一个token之前的序列
            partial_sequence = outputs_gen[:, :-1]
            inputs_extended = {
                'input_ids': partial_sequence,
                'attention_mask': torch.ones_like(partial_sequence, device=partial_sequence.device),
                'pixel_values_videos': inputs_base['pixel_values_videos']
            }
            del outputs_gen
            torch.cuda.empty_cache()
            return self._compute_gradcam_for_position(
                inputs_extended, 0, last_token_id, normalize_method, generated_text, 1
            )
        else:
            print("Warning: No tokens generated")
            num_frames = video.shape[0]
            height, width = video.shape[-2:]
            return torch.zeros(num_frames, height, width), "No generation"

    def _compute_average_token_gradcam(self, inputs_base, video, normalize_method):
        """Compute average GradCAM over all generated tokens with minimal memory."""
        print("Computing average GradCAM over all generated tokens...")
        with torch.no_grad():
            outputs_gen = self.model.generate(
                **inputs_base,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=self.processor.tokenizer.eos_token_id
            )
        input_length = inputs_base['input_ids'].shape[1]
        if outputs_gen.shape[1] <= input_length:
            print("Warning: No tokens generated")
            num_frames = video.shape[0]
            height, width = video.shape[-2:]
            return torch.zeros(num_frames, height, width), "No generation"
        generated_tokens = outputs_gen[0, input_length:]
        generated_text = self.processor.tokenizer.decode(generated_tokens, num_logits_to_keep=True)
        print(f"Generated text: {generated_text}")
        
        # 为每个token计算GradCAM
        all_gradcam_maps = []
        max_tokens = min(100, len(generated_tokens))  # 限制计算量
        
        for i in range(max_tokens):
            token_id = generated_tokens[i]
            position = input_length + i
            print(f"Processing token {i+1}/{max_tokens}: {self.processor.tokenizer.decode(token_id)} (ID: {token_id})")
            
            # 构建到该token为止的序列
            partial_sequence = outputs_gen[:, :position + 1]
            inputs_partial = {
                'input_ids': partial_sequence,
                'attention_mask': torch.ones_like(partial_sequence, device=partial_sequence.device),
                'pixel_values_videos': inputs_base['pixel_values_videos']
            }
            gradcam_map, _ = self._compute_gradcam_for_position(
                inputs_partial, 0, token_id, normalize_method, "", 1
            )
            # Move to CPU and free GPU memory immediately
            gradcam_map_cpu = gradcam_map.detach().cpu()
            del gradcam_map
            torch.cuda.empty_cache()
            if gradcam_map_cpu.sum() > 0:
                all_gradcam_maps.append(gradcam_map_cpu)
        del outputs_gen
        torch.cuda.empty_cache()
        if all_gradcam_maps:
            avg_gradcam = torch.mean(torch.stack(all_gradcam_maps), dim=0)
            return avg_gradcam, f"Average over {len(all_gradcam_maps)} tokens: {generated_text}"
        else:
            num_frames = video.shape[0]
            height, width = video.shape[-2:]
            return torch.zeros(num_frames, height, width), f"Failed to compute average: {generated_text}"

    def _compute_gradcam_for_position(self, inputs, target_position, target_token_id, normalize_method, generated_text, num_logits_to_keep=0):
        """Compute GradCAM for a specific token position, minimizing memory."""
        # Ensure video input requires grad and is on the correct device
        if 'pixel_values_videos' in inputs and inputs['pixel_values_videos'] is not None:
            inputs['pixel_values_videos'] = inputs['pixel_values_videos'].detach().to(self.model.device).requires_grad_(True)
        # No autocast: always use default precision
        outputs = self.model(**inputs, num_logits_to_keep=num_logits_to_keep)
        logits = outputs.logits
        print(f"Logits shape: {logits.shape}")
        print(f"Target position: {target_position}, Target token ID: {target_token_id}")
        if target_position >= logits.shape[1]:
            print(f"Warning: target_position {target_position} >= sequence_length {logits.shape[1]}")
            target_position = logits.shape[1] - 1
        target_logit = logits[0, target_position, target_token_id]
        print(f"Target logit: {target_logit.item():.4f}")
        self.model.zero_grad(set_to_none=True)
        target_logit.backward(retain_graph=False)
        video_gradients = inputs['pixel_values_videos'].grad
        if video_gradients is None:
            print("Warning: No gradients found for video input")
            num_frames = inputs['pixel_values_videos'].shape[1] if len(inputs['pixel_values_videos'].shape) > 1 else 1
            height = inputs['pixel_values_videos'].shape[-2] if len(inputs['pixel_values_videos'].shape) > 2 else 224
            width = inputs['pixel_values_videos'].shape[-1] if len(inputs['pixel_values_videos'].shape) > 3 else 224
            return torch.zeros(num_frames, height, width), generated_text
        print(f"Video gradients shape: {video_gradients.shape}")
        print(f"Video gradients range: [{video_gradients.min().item():.6f}, {video_gradients.max().item():.6f}]")
        
        # 计算GradCAM
        gradcam_maps = self._compute_gradcam_from_gradients(
            video_gradients.detach().cpu(),
            inputs['pixel_values_videos'].detach().cpu(),
            normalize_method=normalize_method
        )
        # Free memory
        del outputs, logits, target_logit, video_gradients
        torch.cuda.empty_cache()
        return gradcam_maps, generated_text

    def _compute_gradcam_from_gradients(
        self,
        gradients: torch.Tensor,
        activations: torch.Tensor,
        normalize_method: str = "per_frame"
    ) -> torch.Tensor:
        """Compute GradCAM from gradients and activations (on CPU)."""
        if gradients.dim() == 5:
            batch_size, num_frames, channels, height, width = gradients.shape
        elif gradients.dim() == 4:
            gradients = gradients.unsqueeze(0)
            activations = activations.unsqueeze(0)
            batch_size, num_frames, channels, height, width = gradients.shape
        else:
            print(f"Unexpected gradient shape: {gradients.shape}")
            return torch.zeros(1, 224, 224)
        
        # 计算权重 (全局平均池化)
        weights = torch.mean(gradients.view(batch_size, num_frames, channels, -1), dim=-1)  # [B, F, C]
        
        # 计算加权组合
        gradcam = torch.zeros(batch_size, num_frames, height, width, device=gradients.device, dtype=gradients.dtype)
        
        for i in range(channels):
            gradcam += weights[:, :, i:i+1, None] * activations[:, :, i, :, :]
        
        # ReLU激活
        gradcam = F.relu(gradcam)
        
        print(f"Before normalization - min: {gradcam.min().item():.6f}, max: {gradcam.max().item():.6f}")
        
        # 归一化
        if normalize_method == "per_frame":
            for b in range(batch_size):
                for f in range(num_frames):
                    gradcam_frame = gradcam[b, f]
                    gradcam_min = gradcam_frame.min()
                    gradcam_max = gradcam_frame.max()
                    if gradcam_max > gradcam_min:
                        gradcam[b, f] = (gradcam_frame - gradcam_min) / (gradcam_max - gradcam_min)
        elif normalize_method == "global":
            for b in range(batch_size):
                gradcam_batch = gradcam[b]
                gradcam_min = gradcam_batch.min()
                gradcam_max = gradcam_batch.max()
                if gradcam_max > gradcam_min:
                    gradcam[b] = (gradcam_batch - gradcam_min) / (gradcam_max - gradcam_min)
        
        print(f"After {normalize_method} normalization - min: {gradcam.min().item():.6f}, max: {gradcam.max().item():.6f}")
        
        return gradcam[0]

    def compute_frame_importance(
        self,
        video: torch.Tensor,
        question: str,
        method: str = "gradient_norm",
        normalize_method: str = "per_frame"
    ) -> np.ndarray:
        """Compute per-frame importance scores."""
        gradcam_maps, _ = self.compute_video_gradcam(video, question, normalize_method=normalize_method)
        
        if method == "gradient_norm":
            frame_importance = torch.norm(gradcam_maps.view(gradcam_maps.shape[0], -1), dim=1)
        elif method == "gradcam_sum":
            frame_importance = torch.sum(gradcam_maps.view(gradcam_maps.shape[0], -1), dim=1)
        else:
            raise ValueError(f"Unknown method: {method}")
        
        return frame_importance.detach().cpu().numpy()

def get_attention_map(img, att_map, blur=True, overlap=True):
    """
    改进的注意力图叠加函数，参考skimage实现
    """
    try:
        from skimage.filters import gaussian
        from skimage import transform as skimage_transform
    except ImportError:
        print("Warning: skimage not available, using cv2 for resize and no blur")
        # 回退到简单实现
        att_map = att_map.copy()
        att_map -= att_map.min()
        if att_map.max() > 0:
            att_map /= att_map.max()
        att_map = cv2.resize(att_map, (img.shape[1], img.shape[0]))
        
        import matplotlib.pyplot as plt
        cmap = plt.get_cmap("jet")
        att_map_v = cmap(att_map)
        att_map_v = att_map_v[:, :, :3]  # 去掉alpha通道
        
        if overlap:
            att_map_result = (
                1 * (1 - att_map**0.7).reshape(att_map.shape + (1,)) * img
                + (att_map**0.7).reshape(att_map.shape + (1,)) * att_map_v
            )
        else:
            att_map_result = att_map_v
            
        return att_map_result
    
    # 使用skimage的完整实现
    att_map = att_map.copy()
    att_map -= att_map.min()
    if att_map.max() > 0:
        att_map /= att_map.max()
    
    # 调整到图像尺寸
    att_map = skimage_transform.resize(att_map, img.shape[:2], order=3, mode="constant")
    
    # 高斯模糊
    if blur:
        sigma = 0.02 * max(img.shape[:2])
        att_map = gaussian(att_map, sigma=sigma, preserve_range=True)
        att_map -= att_map.min()
        if att_map.max() > 0:
            att_map /= att_map.max()
    
    # 转换为彩色热力图
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap("jet")
    att_map_v = cmap(att_map)
    att_map_v = att_map_v[:, :, :3]  # 去掉alpha通道
    
    if overlap:
        # 使用非线性混合
        att_map_result = (
            1 * (1 - att_map**0.7).reshape(att_map.shape + (1,)) * img
            + (att_map**0.7).reshape(att_map.shape + (1,)) * att_map_v
        )
    else:
        att_map_result = att_map_v
        
    return att_map_result

def visualize_video_gradcam_enhanced(
    gradcam_maps: torch.Tensor,
    original_video: np.ndarray,
    output_path: str = "gradcam_frames",
    save_grid: bool = True,
    grid_rows: int = 2,
    grid_cols: int = 4,
    normalize_method: str = "per_frame",
    preserve_scale: bool = False  # 新增参数：是否保持原始数值尺度
):
    """
    Enhanced visualization function for GradCAM results.
    """
    import matplotlib.pyplot as plt
    import os

    os.makedirs(output_path, exist_ok=True)
    
    # 确保gradcam_maps是numpy
    if isinstance(gradcam_maps, torch.Tensor):
        gradcam_maps = gradcam_maps.detach().cpu().numpy()
    
    num_frames = min(gradcam_maps.shape[0], original_video.shape[0])
    
    print(f"Original video shape: {original_video.shape}")
    print(f"GradCAM maps shape: {gradcam_maps.shape}")
    print(f"Video dtype: {original_video.dtype}, range: [{original_video.min()}, {original_video.max()}]")
    print(f"GradCAM range: [{gradcam_maps.min():.4f}, {gradcam_maps.max():.4f}]")
    print(f"Using normalize method: {normalize_method}")
    
    # 保存单独的帧图像
    for i in range(num_frames):
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # 原始帧处理
        orig_frame = original_video[i]
        
        if orig_frame.dtype == np.uint8:
            orig_frame_display = orig_frame.astype(np.float32) / 255.0
        elif orig_frame.max() > 1.0:
            orig_frame_display = np.clip(orig_frame / 255.0, 0, 1)
        else:
            orig_frame_display = np.clip(orig_frame, 0, 1)
        
        axes[0].imshow(orig_frame_display)
        axes[0].set_title(f'Original Frame {i}', fontsize=12)
        axes[0].axis('off')
        
        # GradCAM热力图
        gradcam_frame = gradcam_maps[i]
        
        if gradcam_frame.shape != orig_frame.shape[:2]:
            gradcam_frame = cv2.resize(gradcam_frame, (orig_frame.shape[1], orig_frame.shape[0]))
        
        im1 = axes[1].imshow(gradcam_frame, cmap='jet', vmin=0, vmax=1)
        axes[1].set_title(f'GradCAM Frame {i}', fontsize=12)
        axes[1].axis('off')
        plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
        
        # 叠加图 - 使用改进的方法
        overlay = get_attention_map(orig_frame_display, gradcam_frame, blur=True, overlap=True)
        
        axes[2].imshow(overlay)
        axes[2].set_title(f'Overlay Frame {i}', fontsize=12)
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f'gradcam_frame_{i:04d}.png'),
                   dpi=150, bbox_inches='tight')
        plt.close()
    
    # 创建集中展示图
    if save_grid:
        create_overlay_grid(gradcam_maps, original_video, output_path, grid_rows, grid_cols,
                          normalize_method=normalize_method, preserve_scale=preserve_scale)
    
    print(f"GradCAM帧序列已保存到: {output_path}")

def create_overlay_grid(
    gradcam_maps: np.ndarray,
    original_video: np.ndarray,
    output_path: str,
    grid_rows: int = 2,
    grid_cols: int = 4,
    figsize_per_subplot: Tuple[int, int] = (4, 3),
    normalize_method: str = "per_frame",
    preserve_scale: bool = False  # 新增：是否保持原始尺度
):
    """
    Create overlay grid for GradCAM results.
    """
    import matplotlib.pyplot as plt
    
    if isinstance(gradcam_maps, torch.Tensor):
        gradcam_maps = gradcam_maps.detach().cpu().numpy()
    
    num_frames = min(gradcam_maps.shape[0], original_video.shape[0], grid_rows * grid_cols)
    
    print(f"Creating grid with normalize_method: {normalize_method}, preserve_scale: {preserve_scale}")
    
    # 计算总图像大小
    fig_width = figsize_per_subplot[0] * grid_cols + 1
    fig_height = figsize_per_subplot[1] * grid_rows
    
    fig = plt.figure(figsize=(fig_width, fig_height))
    gs = fig.add_gridspec(grid_rows, grid_cols + 1,
                         width_ratios=[1] * grid_cols + [0.05],
                         hspace=0.15, wspace=0.1)
    
    # 存储所有叠加图像
    all_overlays = []
    
    for i in range(grid_rows * grid_cols):
        row = i // grid_cols
        col = i % grid_cols
        
        if i < num_frames:
            # 处理原始帧
            orig_frame = original_video[i]
            
            if orig_frame.dtype == np.uint8:
                orig_frame_display = orig_frame.astype(np.float32) / 255.0
            elif orig_frame.max() > 1.0:
                orig_frame_display = np.clip(orig_frame / 255.0, 0, 1)
            else:
                orig_frame_display = np.clip(orig_frame, 0, 1)
            
            # 处理GradCAM
            gradcam_frame = gradcam_maps[i]
            if gradcam_frame.shape != orig_frame.shape[:2]:
                gradcam_frame = cv2.resize(gradcam_frame, (orig_frame.shape[1], orig_frame.shape[0]))
            
            print(f"Frame {i}: GradCAM range [{gradcam_frame.min():.3f}, {gradcam_frame.max():.3f}]")
            
            # 创建叠加图
            overlay = get_attention_map(orig_frame_display, gradcam_frame, blur=True, overlap=True)
            all_overlays.append((row, col, overlay, gradcam_frame))
    
    # 显示所有叠加图
    vmin, vmax = 0, 1
    for row, col, overlay, gradcam_frame in all_overlays:
        ax = fig.add_subplot(gs[row, col])
        ax.imshow(overlay)
        frame_idx = row * grid_cols + col
        
        # 显示该帧的统计信息
        max_activation = gradcam_frame.max()
        mean_activation = gradcam_frame.mean()
        ax.set_title(f'Frame {frame_idx}\nMax: {max_activation:.2f}\nMean: {mean_activation:.2f}',
                    fontsize=9, pad=5)
        ax.axis('off')
    
    # 添加空白子图
    for i in range(num_frames, grid_rows * grid_cols):
        row = i // grid_cols
        col = i % grid_cols
        ax = fig.add_subplot(gs[row, col])
        ax.axis('off')
    
    # 创建共享的颜色条
    if all_overlays:
        cbar_ax = fig.add_subplot(gs[:, -1])
        dummy_im = cbar_ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1),
                                 cmap='jet', aspect='auto', vmin=vmin, vmax=vmax)
        
        # 创建颜色条
        cbar_ax.set_xticks([])
        cbar_ax.set_ylabel('Attention Intensity', fontsize=12, rotation=270, labelpad=20)
        
        # 设置颜色条刻度
        cbar_ax.set_yticks(np.linspace(0, 255, 6))
        cbar_ax.set_yticklabels([f'{v:.1f}' for v in np.linspace(vmin, vmax, 6)])
        
        cbar_ax.yaxis.set_label_position('right')
        cbar_ax.yaxis.tick_right()
    
    # 更具体的标题
    title = f'GradCAM Overlay Frames ({normalize_method} normalization)'
    if preserve_scale:
        title += ' - Scale Preserved'
    plt.suptitle(title, fontsize=16, y=0.98)
    
    # 根据normalize_method和preserve_scale生成不同的文件名
    suffix = f"{normalize_method}"
    if preserve_scale:
        suffix += "_preserved"
    
    grid_output_path = os.path.join(output_path, f'overlay_grid_with_colorbar_{suffix}.png')
    plt.savefig(grid_output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"集中展示图（带颜色条，{normalize_method}）已保存到: {grid_output_path}")

def create_gradcam_video(
    gradcam_maps: torch.Tensor,
    original_video: np.ndarray,
    output_path: str,
    fps: int = 30
):
    """Create GradCAM video."""
    if isinstance(gradcam_maps, torch.Tensor):
        gradcam_maps = gradcam_maps.detach().cpu().numpy()
    
    num_frames, height, width = original_video.shape[:3]
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width*2, height))
    
    for i in range(min(num_frames, gradcam_maps.shape[0])):
        orig_frame = original_video[i].copy()
        
        # 确保原始帧范围正确
        if orig_frame.dtype == np.uint8:
            orig_frame_norm = orig_frame.astype(np.float32) / 255.0
        elif orig_frame.max() > 1.0:
            orig_frame_norm = np.clip(orig_frame / 255.0, 0, 1)
        else:
            orig_frame_norm = np.clip(orig_frame, 0, 1)
        
        # 使用改进的叠加方法
        gradcam_frame = gradcam_maps[i]
        overlay_float = get_attention_map(orig_frame_norm, gradcam_frame, blur=True, overlap=True)
        
        # 转换回uint8
        if orig_frame.dtype == np.uint8:
            orig_frame_display = orig_frame
        else:
            orig_frame_display = (orig_frame_norm * 255).astype(np.uint8)
            
        overlay = (overlay_float * 255).astype(np.uint8)
        
        # 并排放置
        combined = np.hstack([orig_frame_display, overlay])
        combined = cv2.cvtColor(combined, cv2.COLOR_RGB2BGR)
        
        out.write(combined)
    
    out.release()
    print(f"GradCAM视频已保存到: {output_path}")

def main():
    from transformers import LlavaOnevisionProcessor, LlavaOnevisionForConditionalGeneration
    import torch

    # Initialize model and processor
    model = LlavaOnevisionForConditionalGeneration.from_pretrained(
        # "llava-hf/llava-onevision-qwen2-0.5b-ov-hf", 
        "llava-hf/llava-onevision-qwen2-7b-ov-hf", 
        torch_dtype=torch.float16, 
        device_map='auto'
    )
    # processor = LlavaOnevisionProcessor.from_pretrained("llava-hf/llava-onevision-qwen2-0.5b-ov-hf")
    processor = LlavaOnevisionProcessor.from_pretrained("llava-hf/llava-onevision-qwen2-7b-ov-hf")
    processor.tokenizer.padding_side = "left"
    
    # 使用你的视频变量
    # video = clip_karate  # shape: (8, 676, 540, 3)
    # video = load_video("4078_ori.mp4") 

    # video_name = "-VqyimSxbpg"
    video_name = "drive-000"
    question_path = "/home/chelly/disk1/Att_VLLM/drive/question.json"

    # original
    # video_path_ori = f"/home/chelly/disk1/Att_VLLM/ROVI/rovi/data/original/{video_name}/1"
    video_path_ori = f"/home/chelly/disk1/Att_VLLM/drive/original/{video_name}/1"
    video = pre_process_video(load_video(video_path_ori))

    # target
    # video_path_tar = f"/home/chelly/disk1/Att_VLLM/ROVI/rovi/data/target/{video_name}"
    # video_path_tar = f"/home/chelly/disk1/Att_VLLM/drive/target/{video_name}"
    # video = pre_process_video(load_video(video_path_tar))

    # attack video
    # video_path_att = f"attack/0.5B-300-default-example/att_video/{video_name}.pt"
    # video_path_att = f"attack/7B-300-default-example/att_video/{video_name}.pt"
    # video_path_att = "/home/chelly/disk1/Att_VLLM/LLaVA-OV/attack/budge/att_video/drive-000-8.pt"
    # video = load_attack_video(video_path_att)
    
    print(f"Input video shape: {video.shape}")
    print(f"Video dtype: {video.dtype}")
    print(f"Video value range: [{video.min()}, {video.max()}]")
    
    # 初始化GradCAM
    gradcam = VideoGradCAM(model, processor)
    
    # 测试三种策略
    # question = "What do you see in this video?"
    question = get_question(question_path, video_name)
    print(f"Question: {question}")

    print("\n=== 测试第一个token策略 ===")
    gradcam_maps_first, answer_first = gradcam.compute_video_gradcam(
        video, question, target_strategy="first_generated", normalize_method="per_frame"
    )
    print(f"First token answer: {answer_first}")
    print(f"First token GradCAM range: [{gradcam_maps_first.min():.4f}, {gradcam_maps_first.max():.4f}]")
    
    # print("\n=== 测试最后一个token策略 ===")
    # gradcam_maps_last, answer_last = gradcam.compute_video_gradcam(
    #     video, question, target_strategy="last_generated", normalize_method="per_frame"
    # )
    # print(f"Last token answer: {answer_last}")
    # print(f"Last token GradCAM range: [{gradcam_maps_last.min():.4f}, {gradcam_maps_last.max():.4f}]")
    
    # print("\n=== 测试平均策略 ===")
    # gradcam_maps_avg, answer_avg = gradcam.compute_video_gradcam(
    #     video, question, target_strategy="answer_average", normalize_method="per_frame"
    # )
    # print(f"Average answer: {answer_avg}")
    # print(f"Average GradCAM range: [{gradcam_maps_avg.min():.4f}, {gradcam_maps_avg.max():.4f}]")
    
    # 可视化比较
    # from your_visualization_code import visualize_video_gradcam_enhanced
    
    visualize_video_gradcam_enhanced(
        gradcam_maps_first, video, "gradcam_first_token", 
        save_grid=True, normalize_method="first_token"
    )
    
    # visualize_video_gradcam_enhanced(
    #     gradcam_maps_last, video, "gradcam_last_token", 
    #     save_grid=True, normalize_method="last_token"
    # )
    
    # visualize_video_gradcam_enhanced(
    #     gradcam_maps_avg, video, "gradcam_average", 
    #     save_grid=True, normalize_method="average"
    # )

if __name__ == "__main__":
    main()


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.60it/s]


Input video shape: (8, 384, 384, 3)
Video dtype: uint8
Video value range: [4, 255]
Question: Is there anything passing through the intersection ahead?

=== 测试第一个token策略 ===
Base input sequence length: 1586
Computing GradCAM for first generated token...
First generated token: No (ID: 2753)
Logits shape: torch.Size([1, 1, 152128])
Target position: 0, Target token ID: 2753


/opt/conda/envs/vlma3/lib/python3.10/site-packages/torch/nn/modules/module.py:1640: FutureWarning: Using non-full backward hooks on a Module that does not return a single Tensor or a tuple of Tensors is deprecated and will be removed in future versions. This hook will be missing some of the grad_output. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
/opt/conda/envs/vlma3/lib/python3.10/site-packages/torch/nn/modules/module.py:1640: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Target logit: 23.5781
Video gradients shape: torch.Size([1, 8, 3, 384, 384])
Video gradients range: [-20.406250, 20.046875]
Before normalization - min: 0.000000, max: 0.000009
After per_frame normalization - min: 0.000000, max: 1.000000
First token answer: No
First token GradCAM range: [0.0000, 1.0000]
Original video shape: (8, 384, 384, 3)
GradCAM maps shape: (8, 384, 384)
Video dtype: uint8, range: [4, 255]
GradCAM range: [0.0000, 1.0000]
Using normalize method: first_token
Creating grid with normalize_method: first_token, preserve_scale: False
Frame 0: GradCAM range [0.000, 1.000]
Frame 1: GradCAM range [0.000, 1.000]
Frame 2: GradCAM range [0.000, 1.000]
Frame 3: GradCAM range [0.000, 1.000]
Frame 4: GradCAM range [0.000, 1.000]
Frame 5: GradCAM range [0.000, 1.000]
Frame 6: GradCAM range [0.000, 1.000]
Frame 7: GradCAM range [0.000, 1.000]
集中展示图（带颜色条，first_token）已保存到: gradcam_first_token/overlay_grid_with_colorbar_first_token.png
GradCAM帧序列已保存到: gradcam_first_token
